# Synthetic Context Contrast VAD Analysis

This notebook tests whether valence, arousal, and dominance (VAD) scores change when the same focal news event is embedded in opposite emotional contexts.

Flow:
1) loads the synthetic paired dataset from `data/synthetic`
2) loads the project VAD model
3) scores each contextualized news text
4) summarizes VAD by context polarity
5) computes pair-level deltas between positive and negative contexts
6) exports row-level and summary CSVs

Default model: `RobroKools/vad-bert`.
Note: the first run may download the model weights from Hugging Face.

## 1) Imports

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from misinformation_simulation.audits import (
    DEFAULT_VAD_MODEL_NAME,
    VAD_DIMENSIONS,
    annotate_vad_scores,
    load_huggingface_vad_model,
    summarize_vad_by_group,
)

## 2) Configuration

In [ ]:
DATA_PATH = Path("../data/synthetic/vad_context_contrast_news.csv")
OUTPUT_DIR = Path("../output/audit/VADContextContrastAudit")
TEXT_COLUMN = "article_text"
PAIR_COLUMN = "pair_id"
CONTEXT_COLUMN = "context_polarity"
MODEL_NAME = DEFAULT_VAD_MODEL_NAME
BATCH_SIZE = 16
SHOW_PROGRESS = True
TOP_N_PAIRS = 10

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH: {DATA_PATH.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"MODEL_NAME: {MODEL_NAME}")

## 3) Load Synthetic Pairs

In [ ]:
context_df = pd.read_csv(DATA_PATH)

expected_columns = {
    PAIR_COLUMN,
    "topic",
    CONTEXT_COLUMN,
    "focal_event",
    "context_frame",
    TEXT_COLUMN,
}
missing_columns = expected_columns.difference(context_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

pair_counts = context_df.groupby(PAIR_COLUMN)[CONTEXT_COLUMN].nunique()
if (pair_counts != 2).any():
    raise ValueError("Each pair must contain exactly two context polarity variants.")

print(f"Rows: {len(context_df)}")
print(f"Pairs: {context_df[PAIR_COLUMN].nunique()}")
display(context_df.head(6))

## 4) Score VAD

In [ ]:
vad_model = load_huggingface_vad_model(MODEL_NAME)

vad_df = annotate_vad_scores(
    context_df,
    text_column=TEXT_COLUMN,
    model_bundle=vad_model,
    batch_size=BATCH_SIZE,
    show_progress=SHOW_PROGRESS,
    progress_description="Synthetic context VAD scoring",
)

row_level_output = OUTPUT_DIR / "vad_context_contrast_scored.csv"
vad_df.to_csv(row_level_output, index=False)

display(vad_df.head(6))
print(f"Row-level VAD scores exported to: {row_level_output.resolve()}")

## 5) Context-Level Summary

In [ ]:
context_summary_df = summarize_vad_by_group(vad_df, group_column=CONTEXT_COLUMN)
topic_summary_df = summarize_vad_by_group(vad_df, group_column="topic")

context_summary_output = OUTPUT_DIR / "vad_context_contrast_context_summary.csv"
topic_summary_output = OUTPUT_DIR / "vad_context_contrast_topic_summary.csv"
context_summary_df.to_csv(context_summary_output, index=False)
topic_summary_df.to_csv(topic_summary_output, index=False)

display(context_summary_df)
display(topic_summary_df)
print(f"Context summary exported to: {context_summary_output.resolve()}")
print(f"Topic summary exported to: {topic_summary_output.resolve()}")

## 6) Pair-Level Deltas

In [ ]:
score_columns = [f"vad_{dimension}" for dimension in VAD_DIMENSIONS]
score_pivot = vad_df.pivot(index=PAIR_COLUMN, columns=CONTEXT_COLUMN, values=score_columns)
metadata_df = (
    vad_df[[PAIR_COLUMN, "topic", "focal_event"]]
    .drop_duplicates(subset=[PAIR_COLUMN])
    .sort_values(PAIR_COLUMN)
    .reset_index(drop=True)
)

pair_delta_df = metadata_df.copy()
delta_columns = []
for dimension in VAD_DIMENSIONS:
    positive_values = score_pivot[(f"vad_{dimension}", "positive")]
    negative_values = score_pivot[(f"vad_{dimension}", "negative")]
    delta_column = f"positive_minus_negative_{dimension}"
    pair_delta_df[f"positive_{dimension}"] = pair_delta_df[PAIR_COLUMN].map(positive_values)
    pair_delta_df[f"negative_{dimension}"] = pair_delta_df[PAIR_COLUMN].map(negative_values)
    pair_delta_df[delta_column] = (
        pair_delta_df[f"positive_{dimension}"] - pair_delta_df[f"negative_{dimension}"]
    )
    pair_delta_df[f"abs_delta_{dimension}"] = pair_delta_df[delta_column].abs()
    delta_columns.append(delta_column)

abs_delta_columns = [f"abs_delta_{dimension}" for dimension in VAD_DIMENSIONS]
pair_delta_df["largest_abs_delta"] = pair_delta_df[abs_delta_columns].max(axis=1)
pair_delta_df["largest_changed_dimension"] = (
    pair_delta_df[abs_delta_columns].idxmax(axis=1).str.replace("abs_delta_", "", regex=False)
)
pair_delta_df = pair_delta_df.sort_values("largest_abs_delta", ascending=False).reset_index(
    drop=True
)

pair_delta_output = OUTPUT_DIR / "vad_context_contrast_pair_deltas.csv"
pair_delta_df.to_csv(pair_delta_output, index=False)

display(pair_delta_df.head(TOP_N_PAIRS))
print(f"Pair-level deltas exported to: {pair_delta_output.resolve()}")

## 7) Inspect Most Sensitive Pairs

In [ ]:
most_sensitive_pair_ids = pair_delta_df.head(TOP_N_PAIRS)[PAIR_COLUMN].tolist()
most_sensitive_examples = vad_df.loc[vad_df[PAIR_COLUMN].isin(most_sensitive_pair_ids)].sort_values(
    [PAIR_COLUMN, CONTEXT_COLUMN]
)[
    [
        PAIR_COLUMN,
        "topic",
        CONTEXT_COLUMN,
        "focal_event",
        TEXT_COLUMN,
        *score_columns,
    ]
]

display(most_sensitive_examples)
print(f"All artifacts saved to: {OUTPUT_DIR.resolve()}")